##Aim:
 To set up Apache Airflow and create a daily scheduled DAG to execute sequential ETL pipeline tasks.

In [ ]:
!python --version

Python 3.12.13


Install Apache Airflow

In [ ]:
!pip install "apache-airflow==3.0.6" \
  --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-3.0.6/constraints-3.12.txt"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.0/152.0 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.5/250.5 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.0/247.0 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━

In [ ]:
!airflow version

3.0.6


In [ ]:
!airflow db migrate

[2026-08-16T05:26:47.929+0000] {providers_manager.py:953} INFO - The hook_class 'airflow.providers.standard.hooks.filesystem.FSHook' is not fully initialized (UI widgets will be missing), because the 'flask_appbuilder' package is not installed, however it is not required for Airflow components to work
[2026-08-16T05:26:47.931+0000] {providers_manager.py:953} INFO - The hook_class 'airflow.providers.standard.hooks.package_index.PackageIndexHook' is not fully initialized (UI widgets will be missing), because the 'flask_appbuilder' package is not installed, however it is not required for Airflow components to work
DB: sqlite:////root/airflow/airflow.db
Performing upgrade to the metadata database sqlite:////root/airflow/airflow.db
[2026-08-16T05:26:48.103+0000] {migration.py:211} INFO - Context impl SQLiteImpl.
[2026-08-16T05:26:48.103+0000] {migration.py:214} INFO - Will assume non-transactional DDL.
[2026-08-16T05:26:48.104+0000] {migration.py:211} INFO - Context impl SQLiteImpl.
[2026-0

In [ ]:
# ============================================================
# Practical No. 6: Setting up Apache Airflow and Creating a DAG
# ============================================================

from datetime import datetime, timedelta
import os

# ------------------------------------------------------------
# Create the Airflow DAGs directory
# ------------------------------------------------------------

dags_folder = "/content/airflow/dags"
os.makedirs(dags_folder, exist_ok=True)


# ------------------------------------------------------------
# Create the DAG Python file
# ------------------------------------------------------------

dag_code = '''
from datetime import datetime, timedelta

from airflow import DAG
from airflow.operators.bash import BashOperator
from airflow.operators.empty import EmptyOperator


# Default arguments for the workflow
default_args = {
    "owner": "data_engineering_lab",
    "depends_on_past": False,
    "start_date": datetime(2026, 1, 1),
    "email_on_failure": False,
    "email_on_retry": False,
    "retries": 1,
    "retry_delay": timedelta(minutes=5),
}


# ------------------------------------------------------------
# Define the DAG
# ------------------------------------------------------------

with DAG(
    dag_id="university_etl_orchestration",
    default_args=default_args,
    description="Daily ETL pipeline orchestration",
    schedule=timedelta(days=1),
    catchup=False,
) as dag:

    # Task 1: Start the pipeline
    start_pipeline = EmptyOperator(
        task_id="start_pipeline"
    )

    # Task 2: Execute the extraction script
    execute_extraction = BashOperator(
        task_id="run_extraction_script",
        bash_command="echo 'Running data extraction script...'"
    )

    # Task 3: Log successful pipeline completion
    pipeline_complete = BashOperator(
        task_id="log_pipeline_success",
        bash_command='echo "ETL Execution completed successfully at $(date)"'
    )


    # --------------------------------------------------------
    # Define task dependencies
    # --------------------------------------------------------

    start_pipeline >> execute_extraction >> pipeline_complete
'''


# ------------------------------------------------------------
# Save the DAG file
# ------------------------------------------------------------

dag_file = os.path.join(dags_folder, "data_pipeline_dag.py")

with open(dag_file, "w") as file:
    file.write(dag_code)


# ------------------------------------------------------------
# Display the created DAG file
# ------------------------------------------------------------

print("Apache Airflow DAG created successfully!")
print("DAG Name: university_etl_orchestration")
print("Schedule: Daily")
print("DAG File:", dag_file)

print("\nTask Sequence:")
print("start_pipeline")
print("      ↓")
print("run_extraction_script")
print("      ↓")
print("log_pipeline_success")

Apache Airflow DAG created successfully!
DAG Name: university_etl_orchestration
Schedule: Daily
DAG File: /content/airflow/dags/data_pipeline_dag.py

Task Sequence:
start_pipeline
      ↓
run_extraction_script
      ↓
log_pipeline_success


Urvi Sawant | 42